In [25]:
# Cell 1: Required imports and basic setup
import numpy as np
import hashlib
import secrets
import hmac
from typing import Dict, List, Tuple, Set, Optional, Any
from dataclasses import dataclass
import random

# Set a fixed seed for reproducibility during testing
# In a real implementation, you would NOT do this
random.seed(42)

In [32]:
# Cell 2: Core wire and gate classes with simplified design

@dataclass
class Wire:
    """Represents a wire in our garbled circuit with two labels (for 0 and 1)"""
    id: str
    label0: bytes  # Random label for value 0
    label1: bytes  # Random label for value 1
    
    @classmethod
    def new(cls, wire_id: str, label_length: int = 16):
        """Create a new wire with random labels"""
        label0 = secrets.token_bytes(label_length)
        label1 = secrets.token_bytes(label_length)
        return cls(wire_id, label0, label1)
    
    def get_label(self, value: int) -> bytes:
        """Get the label corresponding to a value (0 or 1)"""
        if value == 0:
            return self.label0
        elif value == 1:
            return self.label1
        else:
            raise ValueError("Wire value must be 0 or 1")
    
    def get_value_from_label(self, label: bytes) -> int:
        """Get the value corresponding to a label"""
        if hmac.compare_digest(label, self.label0):
            return 0
        elif hmac.compare_digest(label, self.label1):
            return 1
        else:
            raise ValueError(f"Label does not match either value for wire {self.id}")


class GarbledGate:
    """A garbled gate in our circuit"""
    def __init__(self, 
                 gate_type: str,
                 input_wire_a: Wire, 
                 input_wire_b: Wire, 
                 output_wire: Wire,
                 garbled_table: List[Tuple[bytes, bytes, bytes]]):
        self.gate_type = gate_type
        self.input_wire_a = input_wire_a
        self.input_wire_b = input_wire_b
        self.output_wire = output_wire
        self.garbled_table = garbled_table

In [33]:
# Cell 3: Simplified garbling functions for AND and XOR gates

def create_garbled_and_gate(input_wire_a: Wire, input_wire_b: Wire, output_wire: Wire) -> GarbledGate:
    """
    Create a garbled AND gate
    
    Returns a GarbledGate object with a garbled truth table
    """
    # Create a table to store the garbled entries
    # Each entry will be (encrypted_output, tag_a, tag_b)
    garbled_table = []
    
    # For each possible input combination
    for a_val in [0, 1]:
        for b_val in [0, 1]:
            # Compute the actual gate output
            out_val = a_val & b_val
            
            # Get the corresponding wire labels
            label_a = input_wire_a.get_label(a_val)
            label_b = input_wire_b.get_label(b_val)
            out_label = output_wire.get_label(out_val)
            
            # Create encryption key from input labels
            encryption_key = hashlib.sha256(label_a + label_b).digest()[:len(out_label)]
            
            # Encrypt the output label
            encrypted_output = bytes(o ^ k for o, k in zip(out_label, encryption_key))
            
            # Add to the table
            garbled_table.append((encrypted_output, label_a, label_b))
    
    # Shuffle the table for security
    random.shuffle(garbled_table)
    
    return GarbledGate("AND", input_wire_a, input_wire_b, output_wire, garbled_table)


def create_garbled_xor_gate(input_wire_a: Wire, input_wire_b: Wire, output_wire: Wire) -> GarbledGate:
    """
    Create a garbled XOR gate
    
    Returns a GarbledGate object with a garbled truth table
    """
    # Create a table to store the garbled entries
    garbled_table = []
    
    # For each possible input combination
    for a_val in [0, 1]:
        for b_val in [0, 1]:
            # Compute the actual gate output
            out_val = a_val ^ b_val
            
            # Get the corresponding wire labels
            label_a = input_wire_a.get_label(a_val)
            label_b = input_wire_b.get_label(b_val)
            out_label = output_wire.get_label(out_val)
            
            # Create encryption key from input labels
            encryption_key = hashlib.sha256(label_a + label_b).digest()[:len(out_label)]
            
            # Encrypt the output label
            encrypted_output = bytes(o ^ k for o, k in zip(out_label, encryption_key))
            
            # Add to the table
            garbled_table.append((encrypted_output, label_a, label_b))
    
    # Shuffle the table for security
    random.shuffle(garbled_table)
    
    return GarbledGate("XOR", input_wire_a, input_wire_b, output_wire, garbled_table)

In [34]:
# Cell 4: Evaluation function with improved error handling and debugging

def evaluate_garbled_gate(input_label_a: bytes, input_label_b: bytes, garbled_gate: GarbledGate, 
                          debug: bool = False) -> bytes:
    """
    Evaluate a garbled gate with the given input labels.
    
    Args:
        input_label_a: Label for the first input wire
        input_label_b: Label for the second input wire
        garbled_gate: The garbled gate to evaluate
        debug: If True, prints debugging information
        
    Returns:
        The output label
    """
    if debug:
        print(f"Evaluating {garbled_gate.gate_type} gate")
        print(f"Input label A: {input_label_a.hex()[:8]}...")
        print(f"Input label B: {input_label_b.hex()[:8]}...")
    
    # Try each entry in the garbled table
    for i, (encrypted_output, tag_a, tag_b) in enumerate(garbled_gate.garbled_table):
        if debug:
            print(f"Trying entry {i+1}/{len(garbled_gate.garbled_table)}")
            print(f"  Tag A: {tag_a.hex()[:8]}...")
            print(f"  Tag B: {tag_b.hex()[:8]}...")
        
        # Check if the input labels match this entry
        if hmac.compare_digest(input_label_a, tag_a) and hmac.compare_digest(input_label_b, tag_b):
            if debug:
                print("  Found matching entry!")
            
            # Create decryption key from input labels
            decryption_key = hashlib.sha256(input_label_a + input_label_b).digest()[:len(encrypted_output)]
            
            # Decrypt the output label
            output_label = bytes(e ^ k for e, k in zip(encrypted_output, decryption_key))
            
            if debug:
                print(f"  Decrypted output label: {output_label.hex()[:8]}...")
            
            return output_label
    
    # If we reach this point, no matching entry was found
    raise ValueError(f"Failed to evaluate {garbled_gate.gate_type} gate: No matching entry found for the given input labels")

In [35]:
# Cell 5: Test functions with clear output and diagnostics

def test_garbled_gate(gate_type: str, debug: bool = False):
    """Test a garbled gate with all possible inputs"""
    
    # Create wires
    wire_a = Wire.new("a")
    wire_b = Wire.new("b")
    wire_out = Wire.new("out")
    
    # Create the garbled gate
    if gate_type == "AND":
        garbled_gate = create_garbled_and_gate(wire_a, wire_b, wire_out)
    elif gate_type == "XOR":
        garbled_gate = create_garbled_xor_gate(wire_a, wire_b, wire_out)
    else:
        raise ValueError(f"Unsupported gate type: {gate_type}")
    
    print(f"\nTesting garbled {gate_type} gate:")
    
    # Test all possible input combinations
    for a_val in [0, 1]:
        for b_val in [0, 1]:
            # Compute expected output
            if gate_type == "AND":
                expected_out_val = a_val & b_val
            else:  # XOR
                expected_out_val = a_val ^ b_val
            
            # Get input labels
            a_label = wire_a.get_label(a_val)
            b_label = wire_b.get_label(b_val)
            
            # Get expected output label
            expected_out_label = wire_out.get_label(expected_out_val)
            
            # Evaluate garbled gate
            try:
                if debug:
                    print(f"\nEvaluating {gate_type} gate with inputs a={a_val}, b={b_val}")
                    print(f"Expected output: {expected_out_val}")
                
                output_label = evaluate_garbled_gate(a_label, b_label, garbled_gate, debug=debug)
                
                # Check if output matches expected
                is_correct = hmac.compare_digest(output_label, expected_out_label)
                
                if is_correct:
                    print(f"✅ {gate_type}({a_val}, {b_val}) = {expected_out_val} - CORRECT")
                else:
                    print(f"❌ {gate_type}({a_val}, {b_val}) - FAILED! Expected {expected_out_label.hex()[:8]}... but got {output_label.hex()[:8]}...")
                
            except Exception as e:
                print(f"❌ {gate_type}({a_val}, {b_val}) - ERROR: {str(e)}")
    
    print(f"{gate_type} gate testing completed.\n")


# Run the tests with debugging for detailed output
print("=== TESTING GARBLED GATES ===")
test_garbled_gate("AND", debug=True)
test_garbled_gate("XOR", debug=False)  # No need for debug output if AND works

=== TESTING GARBLED GATES ===

Testing garbled AND gate:

Evaluating AND gate with inputs a=0, b=0
Expected output: 0
Evaluating AND gate
Input label A: 31d1670e...
Input label B: 8ef61a36...
Trying entry 1/4
  Tag A: 8638a999...
  Tag B: 8ef61a36...
Trying entry 2/4
  Tag A: 31d1670e...
  Tag B: 3ef4e4e9...
Trying entry 3/4
  Tag A: 8638a999...
  Tag B: 3ef4e4e9...
Trying entry 4/4
  Tag A: 31d1670e...
  Tag B: 8ef61a36...
  Found matching entry!
  Decrypted output label: 6ae7df05...
✅ AND(0, 0) = 0 - CORRECT

Evaluating AND gate with inputs a=0, b=1
Expected output: 0
Evaluating AND gate
Input label A: 31d1670e...
Input label B: 3ef4e4e9...
Trying entry 1/4
  Tag A: 8638a999...
  Tag B: 8ef61a36...
Trying entry 2/4
  Tag A: 31d1670e...
  Tag B: 3ef4e4e9...
  Found matching entry!
  Decrypted output label: 6ae7df05...
✅ AND(0, 1) = 0 - CORRECT

Evaluating AND gate with inputs a=1, b=0
Expected output: 0
Evaluating AND gate
Input label A: 8638a999...
Input label B: 8ef61a36...
Trying e

In [36]:
# Cell 6: Security demonstration

def demonstrate_security():
    """Demonstrate the security properties of our garbled circuit implementation"""
    
    print("=== SECURITY DEMONSTRATION ===")
    
    # Create wires for a simple circuit: (a AND b) XOR a
    wire_a = Wire.new("a")
    wire_b = Wire.new("b")
    wire_intermediate = Wire.new("intermediate")
    wire_output = Wire.new("output")
    
    # Create garbled gates
    and_gate = create_garbled_and_gate(wire_a, wire_b, wire_intermediate)
    xor_gate = create_garbled_xor_gate(wire_intermediate, wire_a, wire_output)
    
    # Choose input values
    a_val = 1
    b_val = 0
    
    # Step 1: In a real setting, these labels would be obtained through oblivious transfer
    print("\nSTEP 1: Garbler provides input labels via oblivious transfer")
    a_label = wire_a.get_label(a_val)
    b_label = wire_b.get_label(b_val)
    
    print(f"Input values: a={a_val}, b={b_val} (known to the garbler)")
    print(f"Input label A: {a_label.hex()[:10]}... (evaluator sees only this, not the value)")
    print(f"Input label B: {b_label.hex()[:10]}... (evaluator sees only this, not the value)")
    
    # Step 2: Evaluate the AND gate
    print("\nSTEP 2: Evaluator computes the AND gate")
    intermediate_label = evaluate_garbled_gate(a_label, b_label, and_gate)
    intermediate_val = a_val & b_val  # Actual value (hidden from evaluator)
    
    print(f"Intermediate label: {intermediate_label.hex()[:10]}... (evaluator sees)")
    print(f"Actual intermediate value: {intermediate_val} (hidden from evaluator)")
    
    # Step 3: Evaluate the XOR gate
    print("\nSTEP 3: Evaluator computes the XOR gate")
    output_label = evaluate_garbled_gate(intermediate_label, a_label, xor_gate)
    expected_output_val = intermediate_val ^ a_val
    
    print(f"Output label: {output_label.hex()[:10]}... (evaluator sees)")
    
    # Step 4: Reveal the mapping for the output wire only
    print("\nSTEP 4: Garbler reveals mapping for output wire only")
    output_val = wire_output.get_value_from_label(output_label)
    print(f"Final output value: {output_val}")
    print(f"Verification: (a AND b) XOR a = ({a_val} AND {b_val}) XOR {a_val} = {expected_output_val}")
    
    # Security properties
    print("\nSECURITY PROPERTIES:")
    print("1. The evaluator only sees encrypted labels, not the actual values.")
    print("2. The garbler does not learn the evaluator's inputs (would be protected by oblivious transfer).")
    print("3. The garbled circuit is use-once: a new circuit must be generated for each computation.")
    print("4. Only the final output is revealed, not intermediate values.")

# Run the security demonstration
demonstrate_security()

=== SECURITY DEMONSTRATION ===

STEP 1: Garbler provides input labels via oblivious transfer
Input values: a=1, b=0 (known to the garbler)
Input label A: 4e710234ed... (evaluator sees only this, not the value)
Input label B: 99c45af84b... (evaluator sees only this, not the value)

STEP 2: Evaluator computes the AND gate
Intermediate label: 199f8c8e70... (evaluator sees)
Actual intermediate value: 0 (hidden from evaluator)

STEP 3: Evaluator computes the XOR gate
Output label: 5cbda9fd89... (evaluator sees)

STEP 4: Garbler reveals mapping for output wire only
Final output value: 1
Verification: (a AND b) XOR a = (1 AND 0) XOR 1 = 1

SECURITY PROPERTIES:
1. The evaluator only sees encrypted labels, not the actual values.
2. The garbler does not learn the evaluator's inputs (would be protected by oblivious transfer).
3. The garbled circuit is use-once: a new circuit must be generated for each computation.
4. Only the final output is revealed, not intermediate values.


In [38]:
# Cell 7: Exercise - NOT gate and composite circuit

def create_garbled_not_gate(input_wire: Wire, output_wire: Wire) -> GarbledGate:
    """
    Create a NOT gate using an XOR gate with a constant 1 input
    
    This demonstrates how NOT can be implemented using just XOR gates
    """
    # Create a constant wire with label0 corresponding to value 0 and label1 to value 1
    const_wire = Wire.new("const_one")
    
    # Create an XOR gate with one constant input set to 1
    return create_garbled_xor_gate(input_wire, const_wire, output_wire), const_wire

def create_composite_circuit():
    """Create and evaluate a composite circuit: (a AND b) OR (a XOR b)"""
    print("\n=== COMPOSITE CIRCUIT DEMONSTRATION ===")
    
    # Create input wires
    wire_a = Wire.new("a")
    wire_b = Wire.new("b")
    
    # Create intermediate wires
    wire_and_result = Wire.new("and_result")
    wire_xor_result = Wire.new("xor_result")
    
    # Create output wire
    wire_output = Wire.new("output")
    
    # Create gates
    and_gate = create_garbled_and_gate(wire_a, wire_b, wire_and_result)
    xor_gate = create_garbled_xor_gate(wire_a, wire_b, wire_xor_result)
    
    # For OR, we can use NOT and AND gates with De Morgan's laws
    # OR(x,y) = NOT(AND(NOT(x), NOT(y)))
    
    # Create NOT gates for inputs to the final AND gate
    not_and_gate, const_wire1 = create_garbled_not_gate(wire_and_result, Wire.new("not_and"))
    not_xor_gate, const_wire2 = create_garbled_not_gate(wire_xor_result, Wire.new("not_xor"))
    
    # Create AND gate for NOT(and_result) AND NOT(xor_result)
    nand_gate = create_garbled_and_gate(not_and_gate.output_wire, not_xor_gate.output_wire, Wire.new("nand"))
    
    # Create final NOT gate for the output
    not_final_gate, const_wire3 = create_garbled_not_gate(nand_gate.output_wire, wire_output)
    
    # Now test the circuit with some inputs
    print("Testing composite circuit: (a AND b) OR (a XOR b)")
    for a_val in [0, 1]:
        for b_val in [0, 1]:
            # Expected output
            and_result = a_val & b_val
            xor_result = a_val ^ b_val
            expected_output = and_result | xor_result  # OR operation
            
            print(f"\nInputs: a={a_val}, b={b_val}")
            print(f"Expected output: {expected_output}")
            
            # Evaluate the circuit
            # Get input labels
            a_label = wire_a.get_label(a_val)
            b_label = wire_b.get_label(b_val)
            
            # Evaluate AND gate
            and_result_label = evaluate_garbled_gate(a_label, b_label, and_gate)
            
            # Evaluate XOR gate
            xor_result_label = evaluate_garbled_gate(a_label, b_label, xor_gate)
            
            # Evaluate NOT gates
            const_one_label = const_wire1.get_label(1)  # Label for constant 1
            not_and_label = evaluate_garbled_gate(and_result_label, const_one_label, not_and_gate)
            
            const_one_label = const_wire2.get_label(1)  # Label for constant 1
            not_xor_label = evaluate_garbled_gate(xor_result_label, const_one_label, not_xor_gate)
            
            # Evaluate NAND gate
            nand_label = evaluate_garbled_gate(not_and_label, not_xor_label, nand_gate)
            
            # Evaluate final NOT gate
            const_one_label = const_wire3.get_label(1)  # Label for constant 1
            output_label = evaluate_garbled_gate(nand_label, const_one_label, not_final_gate)
            
            # Get the output value
            output_val = wire_output.get_value_from_label(output_label)
            
            # Verify
            if output_val == expected_output:
                print(f"✅ Circuit evaluation correct: output = {output_val}")
            else:
                print(f"❌ Circuit evaluation INCORRECT: expected {expected_output}, got {output_val}")

# Uncomment to run the composite circuit demonstration
create_composite_circuit()


=== COMPOSITE CIRCUIT DEMONSTRATION ===
Testing composite circuit: (a AND b) OR (a XOR b)

Inputs: a=0, b=0
Expected output: 0
✅ Circuit evaluation correct: output = 0

Inputs: a=0, b=1
Expected output: 1
✅ Circuit evaluation correct: output = 1

Inputs: a=1, b=0
Expected output: 1
✅ Circuit evaluation correct: output = 1

Inputs: a=1, b=1
Expected output: 1
✅ Circuit evaluation correct: output = 1
